# Final FNO model: 1D Burgers (PK-2026-011)
Trains **one** Fourier Neural Operator on **38 training simulations** (190 examples) at 128 points, with **10 simulations** (50 examples) held out for validation.

**Settings:** `modes = 64`, 500 epochs, width 64, 4 Fourier layers, seed 0.
**Training recipe:** Adam, learning rate 1e-3 with cosine decay, weight decay 1e-4, batch size 16, relative L2 loss.
**Output:** the saved model `fno_modes64_38traj_seed0.pt` (weights from the last epoch).

The model learns one step: given the solution at one time (128 numbers), predict the solution at the next time step (128 numbers).


## 1. Setup

In [34]:
import numpy as np
import torch
import torch.nn as nn

def set_seed(seed):
    """Fix the randomness so the run can be repeated."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# Folder that contains train.npz (change if needed)
DATA_DIR = "/Users/vedbhatt/Desktop/AML/learning-physical-dynamics-across-resolutions-main 6.08.58 PM/data"

# Settings
MODES    = 64      # Fourier modes kept in each Fourier layer
WIDTH    = 64      # channels inside the network
N_LAYERS = 4       # number of Fourier layers
EPOCHS   = 600     # max epochs (may stop earlier via early stopping)
SEED     = 0


Running on: cpu


## 2. Load the training data 


In [36]:
data = np.load(f"/Users/vedbhatt/Desktop/aml-group2-main/data/train.npz")

X = torch.tensor(data["field"],  dtype=torch.float32, device=device)   # input:  solution now        (240, 128)
Y = torch.tensor(data["target"], dtype=torch.float32, device=device)   # target: solution next step (240, 128)

print("inputs :", tuple(X.shape))
print("targets:", tuple(Y.shape))


# ── Trajectory-level train / val split ──────────────────────────────────────
# Seed is set HERE (before the permutation) so both FNO and CNN see the
# exact same 38 train / 10 val trajectories when both use SEED=0.
np.random.seed(SEED)
torch.manual_seed(SEED)

n_traj = 48
perm       = np.random.permutation(n_traj)   # shuffle trajectory indices
val_traj   = perm[:10]                        # 10 held-out trajectories
train_traj = perm[10:]                        # 38 training trajectories

# Build flat row indices for each split
val_idx   = np.concatenate([np.arange(t*5, t*5+5) for t in val_traj])
train_idx = np.concatenate([np.arange(t*5, t*5+5) for t in train_traj])

# Subset tensors
X_train, Y_train = X[train_idx], Y[train_idx]   # (190, 128)
X_val,   Y_val   = X[val_idx],   Y[val_idx]     # ( 50, 128)

print(f"Train: {tuple(X_train.shape)}  |  Val: {tuple(X_val.shape)}")


inputs : (240, 128)
targets: (240, 128)
Train: (190, 128)  |  Val: (50, 128)


## 3. Network input: the value and the position of each grid point

In [37]:
def make_input(u):
    """(batch, N) -> (batch, N, 2): the value u and its position x in [0, 1)."""
    batch, n_points = u.shape
    x = torch.linspace(0, 1, n_points + 1, device=u.device)[:-1]
    x = x.expand(batch, n_points)
    return torch.stack([u, x], dim=-1)

## 4. Fourier layer
Converts the signal to waves, keeps the lowest `modes` waves multiplied by learned weights, drops the rest, and converts back.
The weights belong to waves, not to grid points, so the same layer works on any grid size.

In [5]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.modes = modes
        scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):                                   # x: (batch, channels, n_points)
        batch, _, n_points = x.shape
        x_freq = torch.fft.rfft(x)                          # signal -> waves
        out_freq = torch.zeros(batch, self.weights.shape[1], n_points // 2 + 1,
                               dtype=torch.cfloat, device=x.device)
        k = min(self.modes, n_points // 2 + 1)
        out_freq[:, :, :k] = torch.einsum("bix,iox->box",   # lowest k waves x learned weights
                                          x_freq[:, :, :k], self.weights[:, :, :k])
        return torch.fft.irfft(out_freq, n=n_points)        # waves -> signal

## 5. The FNO
Lift to `WIDTH` channels, apply 4 Fourier layers (Fourier path + local path, then GELU), project back to one value per grid point.

In [6]:
class FNO1d(nn.Module):
    def __init__(self, modes=64, width=64, n_layers=4):
        super().__init__()
        self.lift = nn.Linear(2, width)
        self.spec = nn.ModuleList([SpectralConv1d(width, width, modes) for _ in range(n_layers)])
        self.w    = nn.ModuleList([nn.Conv1d(width, width, 1) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, x):                                   # x: (batch, n_points, 2)
        x = self.lift(x).permute(0, 2, 1)
        for fourier, local in zip(self.spec, self.w):
            x = nn.functional.gelu(fourier(x) + local(x))
        x = x.permute(0, 2, 1)
        return self.proj(x).squeeze(-1)                     # (batch, n_points)

## 6. Loss: relative L2 error 

In [7]:
def rel_l2(pred, target):
    error_size  = torch.linalg.vector_norm(pred - target, dim=-1)
    target_size = torch.linalg.vector_norm(target, dim=-1) + 1e-8   # +1e-8 matches CNN baseline
    return (error_size / target_size).mean()


## 7. Train the final model and save it
Up to 600 epochs on 190 training examples (38 trajectories), batch size 32.
Matches CNN baseline: no weight decay, ReduceLROnPlateau (patience 20, factor 0.5), early stopping (patience 40).
Best val-loss checkpoint is saved (not last epoch).


In [8]:
set_seed(SEED)
model = FNO1d(MODES, WIDTH, N_LAYERS).to(device)
print("learnable parameters:", sum(p.numel() for p in model.parameters()))

# No weight decay — matches CNN baseline
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ReduceLROnPlateau — matches CNN baseline (halve LR after 20 bad epochs)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, factor=0.5, patience=20)

train_curve = []
val_curve   = []
best_val    = float('inf')
wait        = 0          # early-stopping counter

for epoch in range(EPOCHS):
    # ── training step ──
    model.train()
    order = torch.randperm(len(X_train), device=device)
    for start in range(0, len(X_train), 32):           # batch size 32 — matches CNN
        idx = order[start:start + 32]
        loss = rel_l2(model(make_input(X_train[idx])), Y_train[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # ── evaluation step ──
    model.eval()
    with torch.no_grad():
        tl = rel_l2(model(make_input(X_train)), Y_train).item()
        vl = rel_l2(model(make_input(X_val)),   Y_val).item()
    train_curve.append(tl)
    val_curve.append(vl)

    # LR schedule step on val loss — matches CNN
    scheduler.step(vl)

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:3d}   train {tl:.6f}   val {vl:.6f}")

    # ── best checkpoint — matches CNN ──
    if vl < best_val - 1e-6:
        best_val = vl
        wait = 0
        torch.save({"state": model.state_dict(),
                    "config": dict(modes=MODES, width=WIDTH, n_layers=N_LAYERS),
                    "seed": SEED, "best_epoch": epoch,
                    "train_hist": train_curve, "val_hist": val_curve,
                    "train_traj": train_traj.tolist(), "val_traj": val_traj.tolist()},
                   f"fno_modes{MODES}_38traj_seed{SEED}.pt")
    else:
        wait += 1

    # ── early stopping (patience 40) — matches CNN ──
    if wait >= 40:
        print(f"Early stop at epoch {epoch}. Best val = {best_val:.6f}")
        break

print(f"\nBest val = {best_val:.6f}  |  saved fno_modes{MODES}_38traj_seed{SEED}.pt")


learnable parameters: 1073857
epoch   0   training error 0.55412
epoch  50   training error 0.01868
epoch 100   training error 0.02586
epoch 150   training error 0.02146
epoch 200   training error 0.01922
epoch 250   training error 0.01467
epoch 300   training error 0.01352
epoch 350   training error 0.00585
epoch 400   training error 0.00312
epoch 450   training error 0.00221
epoch 499   training error 0.00204
saved fno_modes64_all48_seed0.pt


## 8. Load best checkpoint
Load the saved model weights (best val-loss epoch) back into an FNO1d instance.

In [31]:
device_eval = 'cpu'   # evaluation always on CPU for consistency
model_path  = f'/Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/fno_modes64_38traj_seed0.pt'

ckpt = torch.load(model_path, map_location=device_eval, weights_only=False)
model_eval = FNO1d(**ckpt['config']).to(device_eval)
model_eval.load_state_dict(ckpt['state'])
model_eval.eval()
print(f"Loaded {model_path}  (best epoch: {ckpt['best_epoch']})")


Loaded /Users/vedbhatt/Desktop/aml-group2-main/models/FNO_submission/fno_modes64_38traj_seed0.pt  (best epoch: 287)


## 9. Evaluate at 128 / 256 / 512 grid points
The same trained model is applied to all three resolutions **without retraining**.
This works because FNO operates in frequency space — the weights are resolution-agnostic.

In [32]:
def rel_l2_per_sample(pred, target):
    """NumPy version: returns one error value per example."""
    num = np.linalg.norm((pred - target).reshape(len(pred), -1), axis=1)
    den = np.linalg.norm(target.reshape(len(target), -1), axis=1) + 1e-8
    return num / den


In [39]:
DATA_DIR_EVAL = DATA_DIR   # same folder as training data

results = {}
with torch.no_grad():
    for res in [128, 256, 512]:
        data_e = np.load(f"/Users/vedbhatt/Desktop/aml-group2-main/data/eval_{res}.npz")
        X_e = data_e['field'].astype(np.float32)
        Y_e = data_e['target'].astype(np.float32)

        X_t = torch.tensor(X_e, device=device_eval)
        pred = model_eval(make_input(X_t)).cpu().numpy()

        fno_err = rel_l2_per_sample(pred, Y_e)
        id_err  = rel_l2_per_sample(X_e, Y_e)   # identity baseline: predict no change

        results[res] = {'fno': fno_err, 'identity': id_err}

        print(f"=== Resolution {res} ===")
        print(f"  FNO mean rel-L2   : {fno_err.mean():.6f}")
        print(f"  FNO median        : {np.median(fno_err):.6f}")
        print(f"  Identity baseline : {id_err.mean():.6f}")
        print(f"  Improvement factor: {id_err.mean() / fno_err.mean():.2f}x")
        print()


=== Resolution 128 ===
  FNO mean rel-L2   : 0.003440
  FNO median        : 0.000888
  Identity baseline : 0.032534
  Improvement factor: 9.46x

=== Resolution 256 ===
  FNO mean rel-L2   : 0.004266
  FNO median        : 0.001971
  Identity baseline : 0.032549
  Improvement factor: 7.63x

=== Resolution 512 ===
  FNO mean rel-L2   : 0.004495
  FNO median        : 0.002247
  Identity baseline : 0.032552
  Improvement factor: 7.24x



## 10. Save per-example errors
Saves `fno_errors.npz` for use in the bucket evaluation and cross-model comparison plots.

In [40]:
np.savez('fno_errors.npz',
         err_128=results[128]['fno'],
         err_256=results[256]['fno'],
         err_512=results[512]['fno'],
         id_128=results[128]['identity'],
         id_256=results[256]['identity'],
         id_512=results[512]['identity'])
print('Saved per-example errors to fno_errors.npz')

# Summary table
print()
print(f"{'Resolution':<12} {'FNO':>10} {'Identity':>12} {'Improvement':>13}")
print('-' * 50)
for res in [128, 256, 512]:
    fe = results[res]['fno'].mean()
    ie = results[res]['identity'].mean()
    print(f"{res:<12} {fe:>10.6f} {ie:>12.6f} {ie/fe:>12.2f}x")


Saved per-example errors to fno_errors.npz

Resolution          FNO     Identity   Improvement
--------------------------------------------------
128            0.003440     0.032534         9.46x
256            0.004266     0.032549         7.63x
512            0.004495     0.032552         7.24x
